# C3S CMIP6: what needs to be kept?

**Keep the files listed in the `c3s-cmip6` CSV, with their listed versions.**
For directory-based cleanup, the export below lists every version directory
containing these files. Give that list to the storage team as the directories to
exclude from cleanup, including their contents.

- [Intake catalog](https://raw.githubusercontent.com/cp4cds/c3s_34g_manifests/master/intake/catalogs/c3s.yaml)
- [CMIP6 file list currently referenced: `c3s-cmip6_v20260203.csv.gz`](https://raw.githubusercontent.com/cp4cds/c3s_34g_manifests/master/intake/catalogs/c3s-cmip6/c3s-cmip6_v20260203.csv.gz)

The notebook reads the published Intake entry, so the CSV used at runtime is
shown below. This covers **`c3s-cmip6` only**, not the separate decadal collection.
Run with the `c3s-catalogs` conda environment.

In [1]:
from pathlib import Path, PurePosixPath

import intake
import pandas as pd
from IPython.display import FileLink, Markdown, display

cat_url = "https://raw.githubusercontent.com/cp4cds/c3s_34g_manifests/master/intake/catalogs/c3s.yaml"
# Offline, from notebooks/: cat_url = "../intake/catalogs/c3s.yaml"
# Offline, from repository root: cat_url = "intake/catalogs/c3s.yaml"
cat = intake.open_catalog(cat_url)
list(cat)

['c3s-cmip5',
 'c3s-cmip5-daily-pressure-level',
 'c3s-cmip5-daily-single-level',
 'c3s-cmip5-monthly-pressure-level',
 'c3s-cmip5-monthly-single-level',
 'c3s-cmip6',
 'c3s-cmip6-decadal',
 'c3s-cordex',
 'c3s-ipcc-atlas',
 'c3s-cica-atlas-v01',
 'c3s-cica-atlas-v02',
 'c3s-cica-atlas']

In [2]:
source = cat["c3s-cmip6"]
df = source.read()
display(Markdown(
    f"**CSV used:** [{Path(str(source.urlpath)).name}]({source.urlpath})  \n"
    f"**Catalog updated:** {source.metadata.get('last_updated', 'not supplied')}"
))

**CSV used:** [c3s-cmip6_v20260203.csv.gz](https://raw.githubusercontent.com/cp4cds/c3s_34g_manifests/master/intake/catalogs/c3s-cmip6/c3s-cmip6_v20260203.csv.gz)  
**Catalog updated:** 2026-02-03T20:39:00Z

## How much is listed?

The size below is the sum of the CSV's `size` column, treated as bytes. It is
**catalog metadata, not a measurement of the files currently on Lustre**.
Keeping complete directories may retain additional, unlisted files. Their
full disk usage is unknown here.

In [3]:
assert df.path.notna().all() and not df.path.duplicated().any(), "Expected one row per file"
df["size"] = pd.to_numeric(df["size"], errors="raise")
assert df["size"].notna().all() and df["size"].ge(0).all()
assert df["size"].mod(1).eq(0).all(), "Expected integer byte sizes"

# Each parent is the exact version directory from the catalog, not 'latest'.
paths = [PurePosixPath(path) for path in df.path]
assert all(not path.is_absolute() and ".." not in path.parts for path in paths)
assert all(len(path.parts) == 10 and path.parent.name.startswith("v") for path in paths)
df["keep_directory"] = [str(path.parent) for path in paths]
total_bytes = sum(int(size) for size in df["size"])

# Decimal TB and binary TiB describe the same byte total.
display(Markdown(
    f"**Keep {len(df):,} listed files in {df.keep_directory.nunique():,} version directories.**  \n"
    f"They cover **{df.source_id.nunique()} models** from **{df.institution_id.nunique()} institutions**.  \n"
    f"Catalog size: **{total_bytes / 1e12:.2f} TB** = **{total_bytes / 2**40:.2f} TiB** "
    f"({total_bytes:,} bytes)."
))

**Keep 132,997 listed files in 10,357 version directories.**  
They cover **58 models** from **31 institutions**.  
Catalog size: **10.76 TB** = **9.79 TiB** (10,762,175,660,320 bytes).

## Which models and experiments?

One row per institution/model, showing the experiments to retain. This is a
summary of the keep-list; it does **not** mean that the entire model directory
is required. The exported list below identifies the version directories.

In [4]:
models = (
    df.groupby(["institution_id", "source_id"], dropna=False)
    .agg(
        experiments=("experiment_id", lambda values: ", ".join(sorted(values.unique()))),
        files=("path", "count"),
        catalog_bytes=("size", "sum"),
    )
    .rename_axis(index=["Institution", "Model"])
    .rename(columns={"experiments": "Experiments", "files": "Files"})
)
assert models.Files.sum() == len(df)
assert int(models.catalog_bytes.sum()) == total_bytes
model_overview = models.assign(**{"Catalog TB": models.catalog_bytes / 1e12}).drop(columns="catalog_bytes")
model_overview.style.format({"Files": "{:,.0f}", "Catalog TB": "{:.3f}"})

## Give the storage team this directory list

The source root below is the existing CMIP6 pool on Levante. On a Rook VM,
change `/work` to `/mnt/lustre/work`.

The exported text file contains **one absolute version-directory path per line**.
Keep each listed directory **and its contents**, for example:
`…/historical/r1i1p1f1/Amon/tas/gn/v20190710/`.
Do not replace version names with `latest`, or infer that unlisted data can be
deleted: this list only states what this CDS collection needs.

In [5]:
source_root = PurePosixPath("/work/ik1017/CMIP6/data/CMIP6")
assert source_root.is_absolute()
keep_directories = sorted(set(df.keep_directory))
absolute_directories = [str(source_root / directory) + "/" for directory in keep_directories]

# Every catalog file must be covered by exactly its listed parent directory.
assert set(df.keep_directory) == set(keep_directories)
print(f"Keep {len(absolute_directories):,} directories and their contents under {source_root}")
print("First five entries (the export contains the complete list):")
print("\n".join(absolute_directories[:5]))

Keep 10,357 directories and their contents under /work/ik1017/CMIP6/data/CMIP6
First five entries (the export contains the complete list):
/work/ik1017/CMIP6/data/CMIP6/CMIP/AS-RCEC/TaiESM1/historical/r1i1p1f1/Amon/clt/gn/v20200623/
/work/ik1017/CMIP6/data/CMIP6/CMIP/AS-RCEC/TaiESM1/historical/r1i1p1f1/Amon/evspsbl/gn/v20200623/
/work/ik1017/CMIP6/data/CMIP6/CMIP/AS-RCEC/TaiESM1/historical/r1i1p1f1/Amon/hfls/gn/v20200623/
/work/ik1017/CMIP6/data/CMIP6/CMIP/AS-RCEC/TaiESM1/historical/r1i1p1f1/Amon/hfss/gn/v20200623/
/work/ik1017/CMIP6/data/CMIP6/CMIP/AS-RCEC/TaiESM1/historical/r1i1p1f1/Amon/hur/gn/v20200623/


In [6]:
# Write handoff files under retention/<collection>/ at the repository root.
# No source data is accessed or changed.
# Support running from either notebooks/ or the repository root.
repo_root = Path(".") if Path("intake/catalogs/c3s.yaml").is_file() else Path("..")
assert (repo_root / "intake/catalogs/c3s.yaml").is_file(), "Run from notebooks/ or the repository root"
export_dir = repo_root / "retention" / "c3s-cmip6"
export_dir.mkdir(parents=True, exist_ok=True)
version_tag = Path(str(source.urlpath)).name.removesuffix(".csv.gz")
keep_list = export_dir / f"{version_tag}_keep-version-directories.txt"
keep_list.write_text("\n".join(absolute_directories) + "\n")
model_csv = export_dir / f"{version_tag}_models.csv"
model_overview.to_csv(model_csv)
notes = export_dir / f"{version_tag}_README.txt"
notes.write_text(
    "C3S CMIP6 retention requirements\n"
    f"Source catalog: {cat_url}\nCSV: {source.urlpath}\n"
    f"Source root: {source_root}\n"
    f"Files listed: {len(df):,}\nVersion directories: {len(keep_directories):,}\n"
    f"Catalog size: {total_bytes:,} bytes ({total_bytes / 1e12:.2f} TB; {total_bytes / 2**40:.2f} TiB)\n\n"
    f"Keep every directory in {keep_list.name}, including its contents.\n"
    "The CSV is the exact file-level requirement, including versions.\n"
    "Directory retention may include extra files; actual disk usage is unknown.\n"
    "This is not a deletion authorization for other data.\n"
    "CMIP6 decadal is a separate collection, not covered here.\n"
    "File existence, recorded sizes and completed retention have not been verified on Lustre.\n"
)
assert keep_list.read_text().splitlines() == absolute_directories
absolute_directory_set = set(absolute_directories)
assert all(str(source_root / path.parent) + "/" in absolute_directory_set for path in paths)
display(Markdown("**Handoff files:** directory keep-list, model summary, and explanation."))
for output in (keep_list, model_csv, notes):
    display(FileLink(str(output)))

**Handoff files:** directory keep-list, model summary, and explanation.

/Users/pingu/Documents/GitHub/cp4cds/c3s_34g_manifests/retention/c3s-cmip6/c3s-cmip6_v20260203_keep-version-directories.txt

/Users/pingu/Documents/GitHub/cp4cds/c3s_34g_manifests/retention/c3s-cmip6/c3s-cmip6_v20260203_models.csv

/Users/pingu/Documents/GitHub/cp4cds/c3s_34g_manifests/retention/c3s-cmip6/c3s-cmip6_v20260203_README.txt

The storage team can use the directory list for retention exclusions.
A separate check on Lustre is still needed to confirm that the files exist and
that their sizes match the CSV before retention is considered complete.